Các bước thực hiện trong colab này:
+ Đọc dữ liệu
+ Clean data
+ Xử lý outlier bằng Zscore
+ Cho vào GAN
+ Chia dữ liệu train, test
+ Scale tập train, test
+ Cho vào mô hình
- tỉ lệ trên tập test cân bằng là

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.6 MB/s eta 0:00:00


In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [5]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [6]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1115
1.0,846
-1.0,165


In [7]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1655
2.0,295
3.0,176


Clean data

In [8]:
data.duplicated().sum()

np.int64(13)

In [9]:
data.drop_duplicates(inplace=True)

In [10]:
data.duplicated().sum()

np.int64(0)

In [11]:
data.isna().sum().sum()

np.int64(0)

GAN Zscore

In [12]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from ctgan import CTGAN
from sklearn.model_selection import train_test_split

target_col = 'fetal_health'

print("Kích thước data gốc:", data.shape)
display(data.head())

print("\nPhân bố lớp ban đầu:")
print(data[target_col].value_counts().sort_index())

# =============================
# 1. Xử lý outlier bằng Z-score
# =============================
X_raw = data.drop(columns=[target_col])
y_raw = data[target_col]

# Chỉ lấy các cột số
num_cols = X_raw.select_dtypes(include=['int64', 'float64']).columns

# Tính Z-score
z_scores = np.abs(zscore(X_raw[num_cols], nan_policy='omit'))

# Chuyển về DataFrame để giữ index
z_scores = pd.DataFrame(z_scores, columns=num_cols, index=X_raw.index)

# Ngưỡng Z-score
threshold = 3.5

# Giữ lại các dòng không chứa outlier
mask = (z_scores < threshold).all(axis=1)

X_clean = X_raw[mask]
y_clean = y_raw[mask]

# Ghép lại thành dataframe sạch
data_clean = pd.concat([X_clean, y_clean], axis=1).reset_index(drop=True)

print("\nKích thước sau khi xử lý Z-score:", data_clean.shape)
print("\nPhân bố lớp sau khi xử lý Z-score:")
print(data_clean[target_col].value_counts().sort_index())

# =============================
# 2. Train CTGAN trên data đã làm sạch
# =============================
gan_df = data_clean.copy()

ctgan = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan.fit(gan_df, discrete_columns=[target_col])

print("\nĐã train xong CTGAN")

# =============================
# 3. Sinh thêm dữ liệu để cân bằng
# =============================
class_counts = gan_df[target_col].value_counts()
max_count = class_counts.max()

synthetic_parts = []

for cls, count in class_counts.items():
    need = max_count - count

    if need <= 0:
        continue

    print(f"\nLớp {cls} cần sinh thêm {need} mẫu")

    collected = []
    total_collected = 0

    while total_collected < need:
        sample_n = max(500, need * 2)
        fake_batch = ctgan.sample(sample_n)

        fake_cls = fake_batch[fake_batch[target_col] == cls].copy()

        if len(fake_cls) > 0:
            collected.append(fake_cls)
            total_collected += len(fake_cls)
            print(f"Đã lấy được {total_collected}/{need}")

    fake_cls_final = pd.concat(collected, axis=0).iloc[:need].copy()
    synthetic_parts.append(fake_cls_final)

# =============================
# 4. Gộp dữ liệu thật + synthetic
# =============================
if len(synthetic_parts) > 0:
    synthetic_df = pd.concat(synthetic_parts, axis=0).reset_index(drop=True)
    balanced_df = pd.concat([gan_df, synthetic_df], axis=0).reset_index(drop=True)
else:
    balanced_df = gan_df.copy()

print("\nPhân bố lớp sau CTGAN:")
print(balanced_df[target_col].value_counts().sort_index())

# =============================
# 5. Tách X, y
# =============================
X = balanced_df.drop(columns=[target_col]).copy()
y = balanced_df[target_col].astype(int)

# =============================
# 6. Chia train/test
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nKích thước tập train/test:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nPhân bố y_train:")
print(y_train.value_counts().sort_index())

print("\nPhân bố y_test:")
print(y_test.value_counts().sort_index())

Kích thước data gốc: (2113, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0



Phân bố lớp ban đầu:
fetal_health
1.0    1646
2.0     292
3.0     175
Name: count, dtype: int64

Kích thước sau khi xử lý Z-score: (1920, 22)

Phân bố lớp sau khi xử lý Z-score:
fetal_health
1.0    1554
2.0     279
3.0      87
Name: count, dtype: int64


Gen. (-00.52) | Discrim. (+00.97): 100%|██████████| 300/300 [05:37<00:00,  1.13s/it]



Đã train xong CTGAN

Lớp 2.0 cần sinh thêm 1275 mẫu
Đã lấy được 818/1275
Đã lấy được 1644/1275

Lớp 3.0 cần sinh thêm 1467 mẫu
Đã lấy được 755/1467
Đã lấy được 1473/1467

Phân bố lớp sau CTGAN:
fetal_health
1.0    1554
2.0    1554
3.0    1554
Name: count, dtype: int64

Kích thước tập train/test:
X_train: (3729, 21)
X_test : (933, 21)

Phân bố y_train:
fetal_health
1    1243
2    1243
3    1243
Name: count, dtype: int64

Phân bố y_test:
fetal_health
1    311
2    311
3    311
Name: count, dtype: int64


Scale data

In [13]:
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo scaler
scaler = MinMaxScaler()

# Fit trên train, transform cả train và test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Nếu muốn giữ DataFrame
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print("Đã scale dữ liệu xong")
display(X_train_scaled.head())

Đã scale dữ liệu xong


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
976,0.266821,0.404861,0.021325,0.457979,0.214054,0.462633,0.363527,0.107103,0.336592,0.022807,...,0.370698,0.481521,0.436170,0.149188,0.451326,0.416145,0.518660,0.435233,0.083200,0.534737
675,0.450261,0.206482,0.119897,0.310155,0.214054,0.462633,0.363527,0.714020,0.067621,0.225556,...,0.235428,0.569875,0.322066,0.214635,0.012197,0.463514,0.589054,0.502698,0.058860,0.534737
4635,0.577866,0.222442,0.034838,0.127620,0.231682,0.542279,0.639570,0.976994,0.046153,0.620133,...,0.353265,0.754035,0.333671,0.150547,0.006988,0.451480,0.765075,0.405147,0.024779,0.975551
501,0.366879,0.256077,0.021325,0.655078,0.580086,0.462633,0.363527,0.321309,0.528714,0.022807,...,0.800035,0.055817,0.626342,0.476424,0.012197,0.463514,0.503016,0.452099,0.229239,0.534737
3614,0.439607,0.240150,0.008335,0.635978,0.268872,0.356225,0.898818,0.750129,0.226136,0.027233,...,0.288894,0.249482,0.418243,0.149487,0.010911,0.562142,0.379709,0.551436,0.362808,0.066231


RandomForest

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Khởi tạo model
rf_model_gan = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_model_gan.fit(X_train_scaled, y_train)

# Predict
y_pred = rf_model_gan.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision weighted:", precision_score(y_test, y_pred, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred, average='macro'))
print("\nDetail about one class:")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9421221864951769
Precision weighted: 0.9419227756062806
Recall weighted: 0.9421221864951769
F1 weighted: 0.941996517369418
Precision macro: 0.9419227756062806
Recall macro: 0.9421221864951769
F1 macro: 0.941996517369418

Detail about one class:

Classification Report:
               precision    recall  f1-score   support

           1       0.97      0.98      0.97       311
           2       0.92      0.91      0.91       311
           3       0.94      0.94      0.94       311

    accuracy                           0.94       933
   macro avg       0.94      0.94      0.94       933
weighted avg       0.94      0.94      0.94       933


Confusion Matrix:
 [[304   7   0]
 [ 11 282  18]
 [  0  18 293]]


XgBoot

In [15]:
y_train = y_train.astype(int) - 1
y_test = y_test.astype(int) - 1

In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Khởi tạo model XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',   # dùng cho phân loại nhiều lớp
    num_class=len(y_train.unique()),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

# Train model
xgb_model.fit(X_train_scaled, y_train)

# Predict
y_pred_xg = xgb_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred_xg))
print("Precision weighted:", precision_score(y_test, y_pred_xg, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred_xg, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred_xg, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred_xg, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred_xg, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred_xg, average='macro'))

print("\nDetail about one class:")
print("Classification Report:\n", classification_report(y_test, y_pred_xg))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xg))

Accuracy: 0.9496248660235799
Precision weighted: 0.9496503219868921
Recall weighted: 0.9496248660235799
F1 weighted: 0.9495588453141955
Precision macro: 0.949650321986892
Recall macro: 0.94962486602358
F1 macro: 0.9495588453141955

Detail about one class:
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98       311
           1       0.93      0.91      0.92       311
           2       0.93      0.95      0.94       311

    accuracy                           0.95       933
   macro avg       0.95      0.95      0.95       933
weighted avg       0.95      0.95      0.95       933


Confusion Matrix:
 [[306   5   0]
 [  5 284  22]
 [  0  15 296]]
